In [ ]:
from pathlib import Path

base = Path("/kaggle/input/datasets/anirudrb/stutter-videos")

videos = [
    f for f in base.rglob("*")
    if f.is_file()
    and f.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}
    and not f.name.startswith("._")
    and not f.name.startswith(".")
]

print(f"Total videos found: {len(videos)}")
for v in sorted(videos):
    print(f"  {v.relative_to(base)}")


In [ ]:
import subprocess, sys, os, json
from pathlib import Path

def find_kaggle_bin():
    import shutil
    candidates = [
        shutil.which("kaggle"),
        "/root/.local/bin/kaggle",
        "/usr/local/bin/kaggle",
        str(Path(sys.executable).parent / "kaggle"),
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    print("kaggle CLI not found — installing...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "kaggle", "-q",
         "--no-warn-script-location"],
    )
    bin_dir = str(Path(sys.executable).parent)
    os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")
    for c in [shutil.which("kaggle"),
               str(Path(sys.executable).parent / "kaggle"),
               "/root/.local/bin/kaggle"]:
        if c and Path(c).exists():
            print(f"kaggle CLI installed at: {c}")
            return c
    raise RuntimeError("kaggle binary not found even after install.")

KAGGLE_BIN = find_kaggle_bin()
print(f"Using kaggle binary: {KAGGLE_BIN}")

kaggle_dir = Path(os.path.expanduser("~/.kaggle"))
kaggle_dir.mkdir(exist_ok=True)
cred_path = kaggle_dir / "kaggle.json"
with open(cred_path, "w") as f:
    json.dump({"username": "username", "key": "api-key"}, f)
os.chmod(cred_path, 0o600)
print("Credentials set.")

SEP_DIR = "/kaggle/working/sep28k"
os.makedirs(SEP_DIR, exist_ok=True)

if list(Path(SEP_DIR).rglob("*.wav")):
    print("SEP-28k already downloaded.")
else:
    print("Downloading SEP-28k (~2GB)... please wait.")
    result = subprocess.run(
        [KAGGLE_BIN, "datasets", "download",
         "-d", "ikrbasak/sep-28k", "-p", SEP_DIR, "--unzip"],
        env=os.environ,
    )
    if result.returncode == 0:
        print("Download complete.")
    else:
        raise RuntimeError("SEP-28k download failed — check credentials / internet.")

wavs = list(Path(SEP_DIR).rglob("*.wav"))
csvs = list(Path(SEP_DIR).rglob("*.csv"))
print(f"\nWAV files found : {len(wavs):,}")
print(f"CSV files found : {len(csvs)}")
for c in csvs:
    print(f"  CSV: {c}")


In [ ]:
import subprocess, sys, os
from pathlib import Path

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

packages = [
    "transformers>=4.41.0",
    "torch", "torchaudio", "librosa",
    "audiomentations", "scikit-learn",
    "gradio", "huggingface_hub",
    "seaborn", "matplotlib", "soundfile",
]
for p in packages:
    install(p)

try:
    import torch_xla
    print("torch_xla already available.")
except ImportError:
    print("torch_xla not found.")

os.makedirs("/kaggle/working/stutter_project", exist_ok=True)
os.makedirs("/kaggle/working/stutter_project/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/merged_audio", exist_ok=True)
print("All packages installed.")


In [ ]:
import os
from pathlib import Path

VIDEO_DIR  = "/kaggle/input/datasets/anirudrb/stutter-videos"
SEP_CSV    = "/kaggle/working/sep28k/SEP-28k_labels.csv"
SEP_CLIPS  = "/kaggle/working/sep28k/clips/stuttering-clips/clips"
BASE_DIR   = "/kaggle/working/stutter_project"
MERGED_DIR = "/kaggle/working/merged_audio"
CKPT_DIR   = f"{BASE_DIR}/checkpoints"
ANNOT_PATH = f"{BASE_DIR}/my_annotations.csv"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

print("Verifying paths...")
errors = []
for name, path in [
    ("VIDEO_DIR",  VIDEO_DIR),
    ("SEP_CSV",    SEP_CSV),
    ("SEP_CLIPS",  SEP_CLIPS),
]:
    if Path(path).exists():
        print(f"  OK : {name} -> {path}")
    else:
        errors.append(f"  MISSING: {name} -> {path}")

if errors:
    for e in errors:
        print(e)
    raise FileNotFoundError("Fix missing paths before continuing.")
else:
    print("\nAll paths verified. Ready to continue.")


In [ ]:
import os, random, shutil, subprocess, warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns

from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

from audiomentations import (
    Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift
)

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

# ── TPU setup ─────────────────────────────────────────────────────────────────
import torch_xla
import torch_xla.core.xla_model as xm

# bfloat16 matmuls — TPU v5e-8 is hardware-optimised for bf16
os.environ["XLA_USE_BF16"] = "1"
os.environ["XLA_DYNAMIC_SPMD"] = "0"

device = xm.xla_device()
USE_TPU = True
print(f"Device : TPU -> {device}")

_t = torch.ones(2, 2).to(device)
xm.mark_step()
print(f"TPU smoke test passed.")

TYPE_MAP = {
    "fluent":       0,
    "prolongation": 1,
    "blocking":     2,
    "interjection": 3,
    "repetition":   4,
}
INV_TYPE_MAP = {v: k for k, v in TYPE_MAP.items()}
print("\nConfiguration ready.")


In [ ]:
from pathlib import Path
from collections import defaultdict
import subprocess
import pandas as pd
import soundfile as sf

def extract_audio_from_video(video_path, out_path, sr=16000):
    result = subprocess.run([
        "ffmpeg", "-y", "-i", str(video_path),
        "-ar", str(sr), "-ac", "1", "-vn", str(out_path)
    ], capture_output=True)
    return result.returncode == 0

def validate_audio(path, min_sec=0.5, max_sec=600.0):
    try:
        info = sf.info(str(path))
        return min_sec <= info.duration <= max_sec
    except Exception:
        return False

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}

video_files = [
    f for f in Path(VIDEO_DIR).rglob("*")
    if f.is_file()
    and f.suffix.lower() in VIDEO_EXTENSIONS
    and not f.name.startswith("._")
    and not f.name.startswith(".")
]
print(f"Found {len(video_files)} video files")

by_patient = defaultdict(list)
for f in video_files:
    by_patient[f.parent.name].append(f.name)
for patient in sorted(by_patient.keys()):
    print(f"  {patient}: {len(by_patient[patient])} video(s)")

extracted, failed = [], []
for vf in sorted(video_files):
    out_stem = f"{vf.parent.name}_{vf.stem}"
    out      = Path(MERGED_DIR) / (out_stem + ".wav")
    if out.exists() and validate_audio(out):
        extracted.append(out_stem)
        continue
    ok = extract_audio_from_video(vf, out)
    if ok and validate_audio(out):
        extracted.append(out_stem)
        info = sf.info(str(out))
        print(f"  Extracted: {vf.parent.name}/{vf.name} ({info.duration:.1f}s)")
    else:
        failed.append(f"{vf.parent.name}/{vf.name}")
        print(f"  FAILED   : {vf.parent.name}/{vf.name}")

print(f"\nExtracted : {len(extracted)}")
print(f"Failed    : {len(failed)}")

template_rows = []
for stem in extracted:
    template_rows.append({
        "filename":      stem + ".wav",
        "stutter_label": 1,
        "stutter_type":  "prolongation",
        "source":        "my_dataset",
        "language":      "kn",
        "filepath":      str(Path(MERGED_DIR) / (stem + ".wav")),
    })

template_df = pd.DataFrame(template_rows)
print(f"Total clips: {len(extracted)}")

if not Path(ANNOT_PATH).exists():
    template_df.to_csv(ANNOT_PATH, index=False)
    print(f"\nTemplate CSV created at: {ANNOT_PATH}")
    print("DOWNLOAD THIS FILE, fill in stutter_type for each clip, then re-upload.")
    print("stutter_type must be one of: prolongation / blocking / interjection / repetition / fluent")
else:
    your_df = pd.read_csv(ANNOT_PATH)
    print(f"Loaded annotations: {len(your_df)} clips")
    invalid = your_df[~your_df["stutter_type"].isin(TYPE_MAP.keys())]
    if len(invalid) > 0:
        print(f"ERROR: Invalid stutter_type in {len(invalid)} rows")
        print(invalid[["filename", "stutter_type"]])
    else:
        print("All labels valid. Ready for Cell 6.")
    print(your_df["stutter_type"].value_counts().to_string())


In [ ]:
import soundfile as sf

def validate_audio_sf(path, min_sec=0.1, max_sec=600.0):
    try:
        info = sf.info(str(path))
        return min_sec <= info.duration <= max_sec
    except Exception:
        return False

SEP_LABEL_COLS = {
    "Prolongation":     "prolongation",
    "Block":            "blocking",
    "SoundRep":         "repetition",
    "WordRep":          "repetition",
    "Interjection":     "interjection",
    "NoStutteredWords": "fluent",
}

print("Loading SEP-28k CSV...")
sep_raw = pd.read_csv(SEP_CSV)
print(f"Total rows: {len(sep_raw):,}")

VALID_LABEL_COLS = {col: label for col, label in SEP_LABEL_COLS.items() if col in sep_raw.columns}
print(f"Label columns: {list(VALID_LABEL_COLS.keys())}")

sep_rows, skipped, not_found, bad_audio = [], 0, 0, 0

for idx, row in sep_raw.iterrows():
    scores = {"prolongation": 0.0, "blocking": 0.0, "repetition": 0.0, "interjection": 0.0, "fluent": 0.0}
    for col, label in VALID_LABEL_COLS.items():
        val = row[col]
        if pd.notna(val):
            scores[label] += float(val)
    if sum(scores.values()) == 0:
        skipped += 1; continue
    best_type  = max(scores, key=scores.get)
    best_count = scores[best_type]
    if best_count < 1:
        skipped += 1; continue
    stutter_label = 0 if best_type == "fluent" else 1
    clip_name     = f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    src           = Path(SEP_CLIPS) / clip_name
    if not src.exists():
        not_found += 1; continue
    if not validate_audio_sf(src):
        bad_audio += 1; continue
    sep_rows.append({
        "filename":      clip_name,
        "stutter_label": stutter_label,
        "stutter_type":  best_type,
        "source":        "sep28k",
        "language":      "en",
        "filepath":      str(src),
    })
    if len(sep_rows) % 5000 == 0 and len(sep_rows) > 0:
        print(f"  {len(sep_rows):,} clips processed...")

sep_df = pd.DataFrame(sep_rows)
print(f"\nUsable clips  : {len(sep_df):,}")
print(f"Skipped       : {skipped:,}")
print(f"Not found     : {not_found:,}")
print(f"Bad audio     : {bad_audio:,}")
print(f"\nType distribution:")
print(sep_df["stutter_type"].value_counts().to_string())

your_df = pd.read_csv(ANNOT_PATH)
your_df["filepath"] = your_df["filename"].apply(lambda x: str(Path(MERGED_DIR) / x))

merged_df = pd.concat([sep_df, your_df], ignore_index=True)
merged_df.to_csv(f"{BASE_DIR}/merged_annotations.csv", index=False)

print(f"\nTotal merged : {len(merged_df):,}")
print(f"  SEP-28k    : {len(sep_df):,}")
print(f"  Your clips : {len(your_df):,}")
print(f"\nType distribution:")
print(merged_df["stutter_type"].value_counts().to_string())


In [ ]:
my_data = merged_df[merged_df["source"] == "my_dataset"]
sep     = merged_df[merged_df["source"] == "sep28k"]

sep_train, sep_val = train_test_split(
    sep, test_size=0.15, stratify=sep["stutter_type"], random_state=42
)

try:
    my_train, my_val = train_test_split(
        my_data, test_size=0.20, stratify=my_data["stutter_type"], random_state=42
    )
    print("Stratified split used.")
except ValueError as e:
    print(f"Falling back to random split: {e}")
    my_train, my_val = train_test_split(my_data, test_size=0.20, random_state=42)

train_df = pd.concat([sep_train, my_train], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.concat([sep_val,   my_val],   ignore_index=True).reset_index(drop=True)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Your clips in train : {len(my_train)}")
print(f"Your clips in val   : {len(my_val)}")
print(f"\nVal type distribution:")
print(val_df["stutter_type"].value_counts().to_string())


In [ ]:
augment_strong = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.020, p=0.6),
    TimeStretch(min_rate=0.80, max_rate=1.20, p=0.5),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
    Shift(min_shift=-0.15, max_shift=0.15, p=0.4),
])

augment_mild = Compose([
    AddGaussianNoise(min_amplitude=0.0005, max_amplitude=0.008, p=0.4),
    TimeStretch(min_rate=0.90, max_rate=1.10, p=0.2),
    PitchShift(min_semitones=-1, max_semitones=1, p=0.2),
])

print("Augmentation ready.")


In [ ]:
# max_sec=4.0 — shorter fixed-length clips for faster XLA tracing on TPU
class MergedStutterDataset(Dataset):

    def __init__(self, df, processor, is_train=True, max_sec=4.0, sr=16000):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.is_train  = is_train
        self.sr        = sr
        self.max_len   = int(max_sec * sr)   # 64,000 — fixed shape every batch

    def __len__(self):
        return len(self.df)

    def _load_and_pad(self, filepath):
        wav, sr = torchaudio.load(str(filepath))
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        wav = wav.mean(dim=0).numpy()
        if len(wav) >= self.max_len:
            start = random.randint(0, len(wav) - self.max_len) if self.is_train                     else (len(wav) - self.max_len) // 2
            wav = wav[start : start + self.max_len]
        else:
            wav = np.pad(wav, (0, self.max_len - len(wav)))
        return wav.astype(np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = Path(row["filepath"]) if "filepath" in row and pd.notna(row["filepath"])                    else None
        if filepath is None or not filepath.exists():
            raise FileNotFoundError(f"File not found: {row.get('filepath')}")

        wav = self._load_and_pad(filepath)

        if self.is_train:
            wav = augment_strong(wav, sample_rate=self.sr) if row["source"] == "my_dataset"                   else augment_mild(wav, sample_rate=self.sr)

        inputs = self.processor(
            wav,
            sampling_rate=self.sr,
            return_tensors="pt",
            padding=False,           # already fixed length — no padding needed
            return_attention_mask=True,
        )

        return {
            "input_values": inputs.input_values.squeeze(0),
            "binary_label": torch.tensor(int(row["stutter_label"]), dtype=torch.long),
            "type_label":   torch.tensor(TYPE_MAP[row["stutter_type"]], dtype=torch.long),
            "source":       row["source"],
        }


In [ ]:
def build_weighted_sampler(df, my_data_boost=50.0):
    weights = np.where(df["source"].values == "my_dataset", my_data_boost, 1.0).astype(np.float64)
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True,
    )

def build_class_weights(df):
    type_w = compute_class_weight(
        "balanced", classes=np.arange(5),
        y=df["stutter_type"].map(TYPE_MAP).values
    )
    bin_w = compute_class_weight(
        "balanced", classes=np.array([0, 1]),
        y=df["stutter_label"].values
    )
    return (torch.tensor(bin_w,  dtype=torch.float).to(device),
            torch.tensor(type_w, dtype=torch.float).to(device))

print("Sampler ready.")


In [ ]:
from transformers import Wav2Vec2Model
import torch, torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean', label_smoothing=0.05):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.reduction       = reduction
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            reduction='none',
            label_smoothing=self.label_smoothing,
        )
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


class StutterClassifier(nn.Module):
    def __init__(self, model_name="facebook/wav2vec2-base",
                 unfreeze_last_n=4, dropout=0.25):
        super().__init__()
        print(f"Loading {model_name}...")
        self.backbone = Wav2Vec2Model.from_pretrained(model_name)

        # CRITICAL for TPU: disable gradient checkpointing.
        # Without this, PyTorch checkpoint tries getattr(torch, "xla")
        # which raises AttributeError: module 'torch' has no attribute 'xla'
        self.backbone.gradient_checkpointing_disable()

        # Freeze all backbone params
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Unfreeze top N transformer blocks
        total = len(self.backbone.encoder.layers)
        for layer in self.backbone.encoder.layers[total - unfreeze_last_n:]:
            for p in layer.parameters():
                p.requires_grad = True

        for p in self.backbone.feature_projection.parameters():
            p.requires_grad = True
        for p in self.backbone.encoder.layer_norm.parameters():
            p.requires_grad = True

        hidden = self.backbone.config.hidden_size   # 768 for wav2vec2-base

        self.attn_pool = nn.Sequential(
            nn.Linear(hidden, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.shared = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        self.binary_head = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 2),
        )
        self.type_head = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 5),
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_p   = sum(p.numel() for p in self.parameters())
        print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.1f}%)")
        print(f"Gradient checkpointing disabled: {not self.backbone.is_gradient_checkpointing}")

    def forward(self, input_values, attention_mask=None):
        out    = self.backbone(input_values, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        scores  = self.attn_pool(hidden).squeeze(-1)
        weights = torch.softmax(scores, dim=-1)
        pooled  = (hidden * weights.unsqueeze(-1)).sum(dim=1)
        shared  = self.shared(pooled)
        return self.binary_head(shared), self.type_head(shared)


model = StutterClassifier(model_name="facebook/wav2vec2-base", unfreeze_last_n=4)
model = model.to(device)
print(f"Model on: {device}")


In [ ]:
from transformers import Wav2Vec2FeatureExtractor

processor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
print("Processor ready.")

train_ds = MergedStutterDataset(train_df, processor, is_train=True)
val_ds   = MergedStutterDataset(val_df,   processor, is_train=False)

BATCH_SIZE  = 16   # larger batch — wav2vec2-base fits well on TPU HBM
NUM_WORKERS = 0    # must be 0 on TPU

sampler = build_weighted_sampler(train_df, my_data_boost=50.0)

train_loader = DataLoader(
    train_ds,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = NUM_WORKERS,
    pin_memory  = False,
    drop_last   = True,   # critical — prevents shape retracing on last batch
)
val_loader = DataLoader(
    val_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = False,
    drop_last   = False,
)

print(f"Batch size    : {BATCH_SIZE}")
print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Train samples : {len(train_ds):,}")
print(f"Val samples   : {len(val_ds):,}")


In [ ]:
import math

bin_w, type_w = build_class_weights(train_df)

binary_loss_fn = FocalLoss(weight=bin_w,  gamma=2.0, label_smoothing=0.05)
type_loss_fn   = FocalLoss(weight=type_w, gamma=2.5, label_smoothing=0.05)

backbone_params = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
head_params     = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]

optimizer = AdamW([
    {"params": backbone_params, "lr": 5e-5,  "weight_decay": 0.01},
    {"params": head_params,     "lr": 2e-4,  "weight_decay": 0.01},
], eps=1e-6)

EPOCHS         = 15
PATIENCE       = 4
WARMUP_EPOCHS  = 2
LOG_EVERY_N    = 50    # print every 50 steps
BINARY_LOSS_WT = 0.40
TYPE_LOSS_WT   = 0.60

def warmup_cosine_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine_lambda)

print("Training setup complete.")
print(f"Device : {device}  |  Batch : {BATCH_SIZE}  |  Epochs max : {EPOCHS}")
print(f"Model  : wav2vec2-base | Audio : 4s clips | LOG_EVERY_N : {LOG_EVERY_N}")
print(f"PATIENCE : {PATIENCE} | Loss weights : binary={BINARY_LOSS_WT} type={TYPE_LOSS_WT}")


In [ ]:
def evaluate(loader, model, device):
    model.eval()
    all_b_preds, all_b_labels = [], []
    all_t_preds, all_t_labels = [], []
    all_sources               = []
    total_loss = 0.0
    n_batches  = 0

    with torch.no_grad():
        for batch in loader:
            x     = batch["input_values"].to(device)
            b_lbl = batch["binary_label"].to(device)
            t_lbl = batch["type_label"].to(device)
            b_logits, t_logits = model(x)
            loss = (BINARY_LOSS_WT * binary_loss_fn(b_logits, b_lbl) +
                    TYPE_LOSS_WT   * type_loss_fn(t_logits,   t_lbl))
            # sync to host for metrics
            xm.mark_step()
            total_loss += loss.item()
            n_batches  += 1
            all_b_preds.extend(b_logits.argmax(1).cpu().numpy())
            all_b_labels.extend(b_lbl.cpu().numpy())
            all_t_preds.extend(t_logits.argmax(1).cpu().numpy())
            all_t_labels.extend(t_lbl.cpu().numpy())
            all_sources.extend(batch["source"])

    all_b_preds  = np.array(all_b_preds)
    all_b_labels = np.array(all_b_labels)
    all_t_preds  = np.array(all_t_preds)
    all_t_labels = np.array(all_t_labels)
    all_sources  = np.array(all_sources)

    binary_acc = accuracy_score(all_b_labels, all_b_preds)
    binary_f1  = f1_score(all_b_labels, all_b_preds, average="macro", zero_division=0)
    type_f1    = f1_score(all_t_labels, all_t_preds, average="macro", zero_division=0)

    my_mask = all_sources == "my_dataset"
    my_acc  = accuracy_score(all_b_labels[my_mask], all_b_preds[my_mask])               if my_mask.sum() > 0 else float("nan")

    return {
        "val_loss":    total_loss / max(1, n_batches),
        "binary_acc":  binary_acc,
        "binary_f1":   binary_f1,
        "type_f1":     type_f1,
        "my_data_acc": my_acc,
    }

print("Evaluation helper ready.")


In [ ]:
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

history = {
    "train_loss": [], "val_loss":   [],
    "binary_acc": [], "binary_f1": [],
    "type_f1":    [], "my_data_acc": [],
}

best_val_score = 0.0
patience_count = 0
start_epoch    = 1

resume_path = f"{CKPT_DIR}/best_model.pt"
if Path(resume_path).exists():
    ckpt = torch.load(resume_path, map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    optimizer.load_state_dict(ckpt["optimizer"])
    best_val_score = ckpt["val_score"]
    start_epoch    = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch} (best score: {best_val_score:.4f})")
else:
    print("Starting fresh training.")

print("=" * 65)
print("TRAINING STARTED")
print(f"SEP-28k: {len(sep_df):,}  |  Your clips: {len(my_data)}")
print(f"Device : {device}  |  USE_TPU: {USE_TPU}")
print("=" * 65)

for epoch in range(start_epoch, EPOCHS + 1):

    model.train()
    epoch_loss = 0.0
    n_logged   = 0
    n_my       = 0

    cur_lr_bb = optimizer.param_groups[0]["lr"]
    cur_lr_hd = optimizer.param_groups[1]["lr"]

    _loader = pl.MpDeviceLoader(train_loader, device)

    for step, batch in enumerate(_loader, 1):
        x     = batch["input_values"].to(device)
        b_lbl = batch["binary_label"].to(device)
        t_lbl = batch["type_label"].to(device)

        n_my += list(batch["source"]).count("my_dataset")

        b_logits, t_logits = model(x)

        bin_loss  = binary_loss_fn(b_logits, b_lbl)
        type_loss = type_loss_fn(t_logits,   t_lbl)
        loss      = BINARY_LOSS_WT * bin_loss + TYPE_LOSS_WT * type_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        xm.optimizer_step(optimizer)
        xm.mark_step()

        if step % LOG_EVERY_N == 0:
            loss_val = loss.item()   # sync here only
            epoch_loss += loss_val
            n_logged   += 1
            print(f"  Ep {epoch:02d} step {step:04d}/{len(train_loader)} "
                  f"| loss {loss_val:.4f} "
                  f"| lr_bb {cur_lr_bb:.2e} | your clips so far: {n_my}")

    avg_loss = epoch_loss / max(1, n_logged)
    scheduler.step()

    metrics    = evaluate(val_loader, model, device)
    binary_acc = metrics["binary_acc"]
    binary_f1  = metrics["binary_f1"]
    type_f1    = metrics["type_f1"]
    val_loss   = metrics["val_loss"]
    my_acc     = metrics["my_data_acc"]

    val_score = 0.40 * binary_f1 + 0.60 * type_f1

    history["train_loss"].append(avg_loss)
    history["val_loss"].append(val_loss)
    history["binary_acc"].append(binary_acc)
    history["binary_f1"].append(binary_f1)
    history["type_f1"].append(type_f1)
    history["my_data_acc"].append(my_acc)

    print(f"\nEpoch {epoch:02d}/{EPOCHS} "
          f"| train_loss {avg_loss:.4f} | val_loss {val_loss:.4f} "
          f"| binary_acc {binary_acc:.3f} | binary_F1 {binary_f1:.3f} "
          f"| type_F1 {type_f1:.3f} "
          f"| your_clips_acc {my_acc:.3f} "
          f"| lr_bb {cur_lr_bb:.2e}")

    if val_score > best_val_score:
        best_val_score = val_score
        patience_count = 0
        torch.save({
            "epoch":      epoch,
            "state_dict": model.state_dict(),
            "optimizer":  optimizer.state_dict(),
            "val_score":  val_score,
            "binary_acc": binary_acc,
            "binary_f1":  binary_f1,
            "type_f1":    type_f1,
            "my_acc":     my_acc,
            "type_map":   TYPE_MAP,
        }, resume_path)
        print(f"  Saved best model (score={val_score:.4f})")
    else:
        patience_count += 1
        print(f"  No improvement ({patience_count}/{PATIENCE})")
        if patience_count >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nTraining complete. Best score: {best_val_score:.4f}")


In [ ]:
FINAL_MODEL_PATH = f"{CKPT_DIR}/final_model.pt"
torch.save({
    "state_dict": model.state_dict(),
    "type_map":   TYPE_MAP,
}, FINAL_MODEL_PATH)
print(f"Final model saved at: {FINAL_MODEL_PATH}")
